# Vett — Data Validation
4 key SQL queries to validate behavioral mismatch patterns

In [1]:
import pandas as pd
import sqlite3

investors = pd.read_csv('investors_v2.csv')
transactions = pd.read_csv('transactions_v2.csv')

conn = sqlite3.connect(':memory:')
investors.to_sql('investors', conn, index=False)
transactions.to_sql('transactions', conn, index=False)

print(f'Loaded {len(investors)} investors and {len(transactions)} transactions')

Loaded 500 investors and 8442 transactions


## Query 1: Self-assessed risk level vs. actual tolerance
Do investors accurately assess their own risk tolerance?

In [2]:
q1 = """
SELECT 
    self_risk_level,
    COUNT(*) AS investor_count,
    ROUND(AVG(actual_tolerance), 1) AS avg_actual_tolerance,
    ROUND(AVG(actual_tolerance) - self_risk_level, 1) AS gap
FROM investors
GROUP BY self_risk_level
ORDER BY self_risk_level
"""

pd.read_sql(q1, conn)

,self_risk_level,investor_count,avg_actual_tolerance,gap
0,1,99,1.9,0.9
1,2,129,2.9,0.9
2,3,127,3.2,0.2
3,4,87,3.0,-1.0
4,5,58,4.2,-0.8


## Query 2: Stated holding period vs. actual holding days
How long do investors actually hold compared to what they say?

In [3]:
q2 = """
SELECT 
    i.stated_horizon,
    COUNT(DISTINCT i.investor_id) AS investor_count,
    ROUND(AVG(t.hold_days), 0) AS avg_actual_hold_days
FROM investors i
JOIN transactions t ON i.investor_id = t.investor_id
WHERE t.hold_days IS NOT NULL
GROUP BY i.stated_horizon
ORDER BY avg_actual_hold_days DESC
"""

pd.read_sql(q2, conn)

,stated_horizon,investor_count,avg_actual_hold_days
0,3-5y,91,143.0
1,1-3y,124,134.0
2,6m-1y,139,130.0
3,5y+,50,122.0
4,<6m,94,114.0


## Query 3: Sell decision reasons
How often do investors panic-sell vs. make rational decisions?

In [4]:
q3 = """
SELECT
    CASE 
        WHEN sell_decision_source = 'panic' THEN 'Panic sell'
        WHEN sell_decision_source = 'follow_others' THEN 'Follow others'
        WHEN sell_decision_source = 'rational_take_profit' THEN 'Rational take profit'
        WHEN sell_decision_source = 'rational_stop_loss' THEN 'Rational stop loss'
        WHEN sell_decision_source = 'need_cash' THEN 'Need cash'
        ELSE sell_decision_source
    END AS sell_reason,
    COUNT(*) AS sell_count,
    ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM transactions WHERE action = 'sell'), 1) AS pct
FROM transactions
WHERE action = 'sell' AND sell_decision_source IS NOT NULL AND sell_decision_source != ''
GROUP BY sell_decision_source
ORDER BY sell_count DESC
"""

pd.read_sql(q3, conn)

,sell_reason,sell_count,pct
0,Rational take profit,1393,36.0
1,Follow others,771,19.9
2,Need cash,705,18.2
3,Panic sell,630,16.3
4,Rational stop loss,372,9.6


## Query 4: Financial literacy vs. external influence on buy decisions
Does financial knowledge reduce herd behavior?

In [5]:
q4 = """
SELECT 
    i.financial_literacy,
    COUNT(*) AS total_buys,
    SUM(CASE WHEN t.decision_source != 'self' THEN 1 ELSE 0 END) AS externally_influenced,
    ROUND(100.0 * SUM(CASE WHEN t.decision_source != 'self' THEN 1 ELSE 0 END) / COUNT(*), 1) AS influenced_pct
FROM investors i
JOIN transactions t ON i.investor_id = t.investor_id
WHERE t.action = 'buy'
GROUP BY i.financial_literacy
ORDER BY influenced_pct DESC
"""

pd.read_sql(q4, conn)

,financial_literacy,total_buys,externally_influenced,influenced_pct
0,low,1529,1293,84.6
1,medium,1622,1120,69.1
2,high,1420,642,45.2
